# Assignment 2 – Road Collision Analysis

**Student ID:** NF1041893

## 1. Load and Inspect

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Support running from project root or from the datasets folder
if (Path("datasets") / "road_collision_2023.csv").exists():
    DATA_DIR = Path("datasets")
else:
    DATA_DIR = Path(".")

road_coll = pd.read_csv(DATA_DIR / "road_collision_2023.csv")
road_veh = pd.read_csv(DATA_DIR / "road_vehicle_2023.csv")

In [ ]:
print("Collision dataset")
print("Shape:", road_coll.shape)
print("\nData types:")
print(road_coll.dtypes)
print("\nFirst 3 rows:")
display(road_coll.head(3))
print("\nMissing values per column:")
print(road_coll.isna().sum())

In [ ]:
print("Vehicle dataset")
print("Shape:", road_veh.shape)
print("\nData types:")
print(road_veh.dtypes)
print("\nFirst 3 rows:")
display(road_veh.head(3))
print("\nMissing values per column:")
print(road_veh.isna().sum())

**Observations:** The collision file has one row per collision (8,160 rows, 44 columns) with identifiers such as `collision_index`, severity, counts, and location codes. The vehicle file is longer (14,558 rows) because each collision can involve multiple vehicles, linked by `collision_index`. The vehicle file has no explicit missing values in the raw CSV, but codes like `-1` for `age_of_driver` represent unknown values. The collision file has missing values in coordinates, `number_of_vehicles`, `number_of_casualties`, and `speed_limit` for some rows.

## 2. Data Cleaning

In [ ]:
# Collision cleaning
road_coll = road_coll.drop_duplicates(subset="collision_index", keep="first")

road_coll["collision_severity"] = pd.to_numeric(road_coll["collision_severity"], errors="coerce")
road_coll.loc[~road_coll["collision_severity"].isin([1, 2, 3]), "collision_severity"] = np.nan

road_coll["speed_limit"] = pd.to_numeric(road_coll["speed_limit"], errors="coerce")
road_coll.loc[road_coll["speed_limit"].isin([0, 5, 150]), "speed_limit"] = np.nan

road_coll["number_of_vehicles"] = pd.to_numeric(road_coll["number_of_vehicles"], errors="coerce")
road_coll["number_of_casualties"] = pd.to_numeric(road_coll["number_of_casualties"], errors="coerce")

# Vehicle cleaning
road_veh["age_of_driver"] = pd.to_numeric(road_veh["age_of_driver"], errors="coerce")
road_veh.loc[road_veh["age_of_driver"] == -1, "age_of_driver"] = np.nan
road_veh.loc[road_veh["age_of_driver"] < 0, "age_of_driver"] = np.nan

road_veh["engine_capacity_cc"] = pd.to_numeric(road_veh["engine_capacity_cc"], errors="coerce")
road_veh.loc[road_veh["engine_capacity_cc"] < 0, "engine_capacity_cc"] = np.nan

road_veh.loc[road_veh["vehicle_type"] == -1, "vehicle_type"] = np.nan

**Data quality issues addressed:**
- **Duplicate collisions:** 160 `collision_index` values appeared twice; duplicates were removed, keeping the first row.
- **Invalid severity:** One record had severity code `9` (not Fatal/Serious/Slight); it was set to NaN as required.
- **Unusual speed limits:** Values `0`, `5`, and `150` mph were treated as invalid and set to NaN.
- **Driver age:** `-1` and other negative ages in the vehicle file were recoded to NaN (unknown age).
- **Engine capacity:** Negative values were set to NaN.
- **Vehicle type:** Code `-1` (unknown) was set to NaN for clearer grouping.

## 3. Filtering and Basic Summaries

In [ ]:
collision_filtered = road_coll[road_coll["collision_severity"].notna()].copy()
print(f"Filtered collisions (valid severity): {len(collision_filtered)}")

top10 = (
    collision_filtered.nlargest(10, "number_of_casualties")[
        [
            "collision_index",
            "police_force",
            "collision_severity",
            "number_of_casualties",
            "number_of_vehicles",
            "speed_limit",
        ]
    ]
)
print("\nTop 10 collisions by number_of_casualties:")
display(top10)

collisions_by_force = collision_filtered["police_force"].value_counts()
print("\nCollisions per police_force (top 15):")
display(collisions_by_force.head(15))

**Interpretation:** After filtering invalid severity, most collisions are slight (severity 3), with fewer serious and fatal events. Casualty counts in the top 10 are relatively high compared with typical single-casualty records. Police force `1` (Metropolitan) records the largest volume of collisions, reflecting the size of that area; counts fall off for smaller forces. Severity in the top-casualty cases is often serious or fatal, showing that high casualty counts align with more severe outcomes.

## 4. Grouped Analysis

In [ ]:
collision_filtered["safety_score"] = 4 - collision_filtered["collision_severity"]

by_force = (
    collision_filtered.groupby("police_force")
    .agg(
        avg_safety_score=("safety_score", "mean"),
        n_collisions=("collision_index", "nunique"),
        total_casualties=("number_of_casualties", "sum"),
    )
    .sort_values("avg_safety_score", ascending=False)
)
print("Grouped by police_force:")
display(by_force.round(2))

by_year_force = (
    collision_filtered.groupby(["collision_year", "police_force"])
    .agg(
        avg_safety_score=("safety_score", "mean"),
        n_collisions=("collision_index", "count"),
    )
    .round(2)
)
print("\nGrouped by collision_year and police_force:")
display(by_year_force)

**Interpretation:** Average safety score is highest where collisions tend to be less severe (more slight, fewer fatal). Forces with fewer collisions can show higher or lower average safety scores depending on local severity mix. Total casualties are driven both by collision volume and by how serious those collisions are. The year–force breakdown shows all records are from 2023 but highlights variation in volume and average severity across forces.

## 5. Filters by Severity and Size

In [ ]:
severe_multi = collision_filtered[
    (collision_filtered["collision_severity"] <= 2)
    & (collision_filtered["number_of_casualties"] >= 2)
]
print(f"Severity <= 2 and at least 2 casualties: {len(severe_multi)}")
display(severe_multi.head())

slight_single = collision_filtered[
    (collision_filtered["collision_severity"] == 3)
    & (collision_filtered["number_of_casualties"] == 1)
]
print(f"\nSeverity = 3 and exactly 1 casualty: {len(slight_single)}")
display(slight_single.head())

high_adj_serious = collision_filtered[
    collision_filtered["collision_adjusted_severity_serious"] >= 0.6
]
print(f"\ncollision_adjusted_severity_serious >= 60%: {len(high_adj_serious)}")
display(high_adj_serious.head())

**Interpretation:** Serious/fatal collisions with two or more casualties are relatively rare compared with the full dataset. The largest group is slight collisions with exactly one casualty—the typical low-severity pattern. Collisions with adjusted serious severity at or above 60% highlight cases where injury outcomes are weighted toward serious harm, even when the recorded collision severity code may differ.

## 6. NumPy Analysis

In [ ]:
speed = collision_filtered["speed_limit"].to_numpy(dtype=float)
casualties = collision_filtered["number_of_casualties"].to_numpy(dtype=float)

finite_mask = np.isfinite(speed) & np.isfinite(casualties)
cov_matrix = np.cov(speed[finite_mask], casualties[finite_mask])
covariance = cov_matrix[0, 1]
print(f"Covariance (speed_limit, number_of_casualties): {covariance:.4f}")

finite_casualties = casualties[np.isfinite(casualties)]
p25, p50, p75 = np.percentile(finite_casualties, [25, 50, 75])
print(f"25th percentile casualties: {p25:.4f}")
print(f"50th percentile casualties: {p50:.4f}")
print(f"75th percentile casualties: {p75:.4f}")

prop_two_plus = np.mean(finite_casualties >= 2)
print(f"Proportion with at least 2 casualties: {prop_two_plus:.4f}")

**Interpretation:** The covariance between speed limit and casualty count is small, suggesting little linear association in this sample. Percentiles show most collisions involve one or two casualties (median 1). Only a modest proportion of collisions have two or more casualties, consistent with many low-casualty slight incidents.

## 7. Derived Column

In [ ]:
collision_filtered["multi_vehicle_collision"] = (
    collision_filtered["number_of_vehicles"].fillna(0) >= 2
).astype(int)

display(
    collision_filtered[
        [
            "collision_index",
            "number_of_vehicles",
            "number_of_casualties",
            "multi_vehicle_collision",
        ]
    ].head()
)

## 8. Vehicle Dataset Analysis

In [ ]:
by_vehicle_type = (
    road_veh.groupby("vehicle_type")
    .agg(
        mean_age_of_driver=("age_of_driver", "mean"),
        n_rows=("collision_index", "count"),
        n_unique_collisions=("collision_index", "nunique"),
    )
    .round(2)
)
print("Grouped by vehicle_type:")
display(by_vehicle_type)

## 9. Merge and Merged Analysis

In [ ]:
merged = collision_filtered.merge(road_veh, on="collision_index", how="inner")

merged["age_of_driver"] = pd.to_numeric(merged["age_of_driver"], errors="coerce")

merged_by_force = (
    merged.groupby("police_force")
    .agg(
        median_age_of_driver=("age_of_driver", "median"),
        std_age_of_driver=("age_of_driver", "std"),
        n_merged_rows=("collision_index", "count"),
        n_vehicle_types=("vehicle_type", "nunique"),
    )
    .sort_values("median_age_of_driver", ascending=False)
    .round(2)
)
print("Merged analysis by police_force:")
display(merged_by_force)

pivot = pd.pivot_table(
    merged,
    index="police_force",
    columns="vehicle_type",
    values="collision_index",
    aggfunc="count",
    fill_value=0,
)
print("\nPivot table (vehicle record counts by police_force and vehicle_type):")
display(pivot.round(2))

**Interpretation:** Median driver age varies by police force, reflecting demographic differences across regions. Standard deviation shows spread in ages within each area. Pivot counts show which vehicle types are most involved in each force—typically passenger cars (type 9) dominate. Larger forces have more vehicle records and a wider mix of vehicle types in the merged data.

## 10. Final Answer and Reproducibility

In [ ]:
summary = (
    collision_filtered.groupby("police_force")
    .agg(
        Avg_Multi_Vehicle_Collision=("multi_vehicle_collision", "mean"),
        N_Collisions=("collision_index", "nunique"),
        Total_Casualties=("number_of_casualties", "sum"),
    )
    .reset_index()
    .sort_values("Avg_Multi_Vehicle_Collision", ascending=False)
)

output_path = DATA_DIR / "analysis_result_road_safety_by_force.csv"
summary.to_csv(output_path, index=False)

print(f"Saved summary to {output_path}")
display(summary.round(4).head(10))

**Conclusion:** Police force **43** has the highest average `multi_vehicle_collision` (approximately 0.80), meaning about four in five of its recorded collisions involve two or more vehicles. The summary table ranks forces by `Avg_Multi_Vehicle_Collision`; force 43 leads that column while also showing its collision count and total casualties for context. Re-running this notebook from the original CSV files regenerates `analysis_result_road_safety_by_force.csv` in the datasets folder.